# Problem 54: RAG with Tables

**Difficulty:** Hard | **Framework:** CrewAI

**Categories:** rag

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/rag-with-tables).

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- Tables in documents must be preserved during chunking
- The agent must correctly answer questions requiring tabular data
- Chunk boundaries must not split table rows apart
- The chunking strategy must handle both prose and tables in the same document


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from crewai import Agent, Task, Crew
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.text_splitter import CharacterTextSplitter

embeddings = OpenAIEmbeddings()

document_text = """Company Financial Report Q3 2024

Revenue Summary:
The company achieved strong growth across all segments.

| Region       | Q3 Revenue | Q2 Revenue | Growth |
|-------------|-----------|-----------|--------|
| North America | $45M      | $38M      | 18.4%  |
| Europe       | $32M      | $29M      | 10.3%  |
| Asia Pacific | $28M      | $22M      | 27.3%  |
| Latin America | $12M      | $11M      | 9.1%   |

Expense Breakdown:
Operating expenses were well managed this quarter.

| Category     | Amount  | % of Revenue |
|-------------|---------|-------------|
| R&D          | $25M    | 21.4%       |
| Sales        | $18M    | 15.4%       |
| Operations   | $15M    | 12.8%       |
| G&A          | $8M     | 6.8%        |

The company projects continued growth in Q4 2024."""

# BUG: Naive character splitting breaks tables apart
splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = splitter.split_text(document_text)
docs = [Document(page_content=chunk) for chunk in chunks]

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

def ask(question: str) -> str:
    retrieved = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in retrieved])
    answerer = Agent(
        role="Financial Analyst",
        goal="Answer questions about financial data",
        backstory=f"Answer based on this context:\n{context}",
    )
    task = Task(description=question, expected_output="A data-backed answer", agent=answerer)
    crew = Crew(agents=[answerer], tasks=[task])
    return str(crew.kickoff())

print(ask("What was the revenue growth in Asia Pacific?"))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Detect table boundaries before chunking and treat each table as an atomic unit
2. Consider using markdown or structured formats to represent tables in chunks
3. Use a custom text splitter that recognizes table delimiters and keeps tables intact


## 7. Evaluation Criteria

Check your solution against these criteria:

- Preserves table structure
- Answers tabular questions correctly
- Handles mixed prose and tables
